# Data Engineering — Moduł 7: Slowly Changing Dimensions (SCD)

Moduł 6 zbudował wymiary (`dim_klienci`, `dim_produkty`) tak, jakby ich atrybuty NIGDY
się nie zmieniały. W rzeczywistości klient PRZEPROWADZA SIĘ do innego miasta, zmienia
segment, produkt zmienia kategorię -- a to, CO ZROBIĆ z taką zmianą w hurtowni, to jedno
z klasycznych pytań modelowania wymiarowego: **Slowly Changing Dimensions (SCD)**.

## Spis treści
1. [Problem: wymiary się zmieniają](#sec1)
2. [SCD Type 0 i Type 1](#sec2)
3. [SCD Type 2 — pełna historia](#sec3)
4. [SCD Type 2 przez `dbt snapshot`](#sec4)
5. [Wzmianka: Type 3 i Type 6](#sec5)
6. [Podsumowanie i ćwiczenia](#sec6)


Generujemy nasz znany zestaw danych sklepu internetowego i ładujemy go do **DuckDB** (lekka, plikowa baza analityczna) jako schemat `raw` -- symulując surowe dane, które w prawdziwym projekcie załadowałby zewnętrzny proces ingestii (Moduł 1). Budujemy też szkielet projektu dbt (`dbt_project.yml`, `profiles.yml`, `sources.yml`) wskazujący na te dane.

In [ ]:
import os
import subprocess
import duckdb
import numpy as np
import pandas as pd

# --- 1. Generujemy surowe dane (ten sam generator co w kursie Databricks/PySpark) ---
rng = np.random.default_rng(7)

n_klienci = 300
miasta = ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segmenty = ["Nowy", "Standardowy", "Premium"]
klienci_pd = pd.DataFrame({
    "klient_id": np.arange(1, n_klienci + 1),
    "miasto": rng.choice(miasta, size=n_klienci, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segmenty, size=n_klienci, p=[0.35, 0.45, 0.20]),
    "dni_od_rejestracji": rng.integers(1, 900, size=n_klienci),
})

n_produkty = 40
kategorie = ["Elektronika", "Odziez", "Dom", "Sport", "Ksiazki"]
srednie_ceny = {"Elektronika": 1200, "Odziez": 150, "Dom": 250, "Sport": 300, "Ksiazki": 50}
kategorie_produktow = rng.choice(kategorie, size=n_produkty)
produkty_pd = pd.DataFrame({
    "produkt_id": np.arange(1, n_produkty + 1),
    "kategoria": kategorie_produktow,
    "cena": [round(max(5.0, rng.normal(srednie_ceny[k], srednie_ceny[k] * 0.3)), 2) for k in kategorie_produktow],
})

n_zamowienia = 3000
klient_id_zamowien = rng.choice(klienci_pd["klient_id"], size=n_zamowienia)
segment_mapa = klienci_pd.set_index("klient_id")["segment"]
segment_zamowien = pd.Series(klient_id_zamowien).map(segment_mapa).values
prawdop_anulowania = np.where(segment_zamowien == "Nowy", 0.22, np.where(segment_zamowien == "Standardowy", 0.10, 0.04))
anulowane = rng.binomial(1, prawdop_anulowania)
status_zamowien = np.where(anulowane == 1, "Anulowane", rng.choice(["Zlozone", "Wyslane", "Dostarczone"], size=n_zamowienia, p=[0.2, 0.3, 0.5]))
data_bazowa = pd.Timestamp("2024-01-01")
dni_offset = rng.integers(0, 730, size=n_zamowienia)
zamowienia_pd = pd.DataFrame({
    "zamowienie_id": np.arange(1, n_zamowienia + 1),
    "klient_id": klient_id_zamowien,
    "data_zamowienia": [data_bazowa + pd.Timedelta(days=int(d)) for d in dni_offset],
    "status": status_zamowien,
})

wiersze_pozycji = []
pozycja_id = 1
cena_produktu = produkty_pd.set_index("produkt_id")["cena"]
for zamowienie_id in zamowienia_pd["zamowienie_id"]:
    liczba_pozycji = rng.integers(1, 5)
    wybrane_produkty = rng.choice(produkty_pd["produkt_id"], size=liczba_pozycji, replace=False)
    for produkt_id in wybrane_produkty:
        ilosc = int(rng.integers(1, 4))
        rabat = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        wiersze_pozycji.append({
            "pozycja_id": pozycja_id, "zamowienie_id": int(zamowienie_id),
            "produkt_id": int(produkt_id), "ilosc": ilosc,
            "cena_jednostkowa": float(cena_produktu[produkt_id]), "rabat": float(rabat),
        })
        pozycja_id += 1
pozycje_zamowien_pd = pd.DataFrame(wiersze_pozycji)

# --- 2. Ladujemy surowe dane do DuckDB, schemat "raw" (jakby zaladowal je zewnetrzny proces ingestii) ---
DBT_DIR = os.path.abspath("dbt_kurs")
for _poddir in ("models", "tests", "macros", "seeds", "snapshots"):
    os.makedirs(f"{DBT_DIR}/{_poddir}", exist_ok=True)
DUCKDB_PATH = f"{DBT_DIR}/kurs_de.duckdb"

con = duckdb.connect(DUCKDB_PATH)
con.execute("CREATE SCHEMA IF NOT EXISTS raw")
con.execute("CREATE OR REPLACE TABLE raw.klienci AS SELECT * FROM klienci_pd")
con.execute("CREATE OR REPLACE TABLE raw.produkty AS SELECT * FROM produkty_pd")
con.execute("CREATE OR REPLACE TABLE raw.zamowienia AS SELECT * FROM zamowienia_pd")
con.execute("CREATE OR REPLACE TABLE raw.pozycje_zamowien AS SELECT * FROM pozycje_zamowien_pd")
con.close()

# --- 3. Szkielet projektu dbt ---
with open(f"{DBT_DIR}/dbt_project.yml", "w") as f:
    f.write(
        "name: kurs_de\n"
        "version: '1.0'\n"
        "profile: kurs_de\n"
        "model-paths: [\"models\"]\n"
        "seed-paths: [\"seeds\"]\n"
        "snapshot-paths: [\"snapshots\"]\n"
    )

with open(f"{DBT_DIR}/profiles.yml", "w") as f:
    f.write(
        "kurs_de:\n"
        "  target: dev\n"
        "  outputs:\n"
        "    dev:\n"
        "      type: duckdb\n"
        f"      path: {DUCKDB_PATH}\n"
    )

with open(f"{DBT_DIR}/models/sources.yml", "w") as f:
    f.write(
        "version: 2\n"
        "sources:\n"
        "  - name: raw_dane\n"
        "    schema: raw\n"
        "    tables:\n"
        "      - name: klienci\n"
        "      - name: produkty\n"
        "      - name: zamowienia\n"
        "      - name: pozycje_zamowien\n"
    )

env = os.environ.copy()
env["DBT_PROFILES_DIR"] = DBT_DIR


def dbt(*args, check=True):
    """Uruchamia komende `dbt ...` w naszym projekcie i wypisuje wynik."""
    wynik = subprocess.run(
        ["dbt", *args, "--project-dir", DBT_DIR],
        env=env, capture_output=True, text=True,
    )
    print(wynik.stdout[-4000:])
    if wynik.returncode != 0:
        print(wynik.stderr[-2000:])
        if check:
            raise RuntimeError(f"dbt {' '.join(args)} zakonczyl sie kodem {wynik.returncode}")
    return wynik


print(f"Zaladowano surowe dane do {DUCKDB_PATH} (schemat raw).")
print(f"Projekt dbt: {DBT_DIR}")


In [ ]:
model_kod = '''
select klient_id, miasto, segment, dni_od_rejestracji
from {{ source('raw_dane', 'klienci') }}
'''
with open(f"{DBT_DIR}/models/stg_klienci.sql", "w") as f:
    f.write(model_kod)

dbt("run", "--select", "stg_klienci")


<a id="sec1"></a>
## 1. Problem: wymiary się zmieniają

Wyobraź sobie: klient #1 mieszkał w Gdańsku, ale właśnie przeprowadził się do Szczecina.
Twoja tabela `dim_klienci` ma JEDEN wiersz per klient -- gdy zaktualizujesz `miasto` na
"Szczecin", co się dzieje z HISTORYCZNYMI zamówieniami tego klienta, złożonymi jeszcze
gdy mieszkał w Gdańsku? Jeśli Twoje zapytanie łączy `fact_sprzedaz` z `dim_klienci` przez
prosty `JOIN`, WSZYSTKIE jego zamówienia (także te sprzed przeprowadzki) nagle "staną
się" zamówieniami ze Szczecina -- fałszując raporty typu "przychód per miasto" dla
przeszłości. **SCD to zestaw strategii radzenia sobie z tym problemem.**

<a id="sec2"></a>
## 2. SCD Type 0 i Type 1

- **Type 0 (Retain Original)** -- NIGDY nie aktualizuj wartości po pierwszym załadowaniu,
  nawet jeśli źródło się zmieni. Rzadko używane dla całych wymiarów, ale częste dla
  POJEDYNCZYCH kolumn, które z definicji są niezmienne (np. `data_pierwszego_zamowienia`
  klienta -- fakt historyczny, który NIGDY nie powinien się zmienić).
- **Type 1 (Overwrite)** -- NADPISZ starą wartość nową, bez śladu historii. Najprostsze w
  implementacji (zwykły `UPDATE` albo, w dbt, zwykły model `table`/`view` czytający
  BIEŻĄCY stan źródła -- dokładnie to, co robiliśmy w Module 6!). Odpowiednie, gdy
  historia NIE MA znaczenia biznesowego (np. poprawka literówki w nazwisku klienta).

In [ ]:
# SCD Type 1 w praktyce -- to jest DOKLADNIE nasz model dim_klienci z Modulu 6.
# Kazde uruchomienie 'dbt run' NADPISUJE poprzedni stan biezacym stanem zrodla --
# historia poprzednich wartosci miasto/segment jest NIEODWRACALNIE utracona.
con = duckdb.connect(DUCKDB_PATH)
print("Stan PRZED zmiana:", con.execute("SELECT miasto FROM main.stg_klienci WHERE klient_id = 1").fetchone())

con.execute("UPDATE raw.klienci SET miasto = 'Szczecin' WHERE klient_id = 1")
con.close()

dbt("run", "--select", "stg_klienci")

con = duckdb.connect(DUCKDB_PATH)
print("Stan PO zmianie (Type 1 -- stare Gdansk ZNIKNELO):", con.execute("SELECT miasto FROM main.stg_klienci WHERE klient_id = 1").fetchone())
con.close()


> ⚠️ **Type 1 to DOMYŚLNE zachowanie zwykłych modeli dbt**
>
> Jeśli nie zrobisz NIC specjalnego (jak w Module 6), KAŻDY model dbt czytający bieżący stan źródła jest, DE FACTO, wymiarem typu SCD 1 -- historia poprzednich wartości po prostu nie istnieje. To OK dla wielu atrybutów, ale problematyczne dla tych, które wpływają na raportowanie historyczne (miasto, segment, cena katalogowa produktu w momencie sprzedaży).

<a id="sec3"></a>
## 3. SCD Type 2 — pełna historia

**Type 2** rozwiązuje problem z sekcji 1: zamiast NADPISYWAĆ wiersz, DODAJE NOWY wiersz
dla zmienionej wartości, a stary wiersz oznacza jako "nieaktualny". Typowe kolumny
techniczne:

- **`dbt_valid_from`** -- od kiedy ten wiersz reprezentuje aktualny stan.
- **`dbt_valid_to`** -- do kiedy był aktualny (`NULL`/`NaT` dla WCIĄŻ aktualnego wiersza).
- (czasem) **`is_current`** -- flaga `True`/`False` zamiast sprawdzania `dbt_valid_to
  IS NULL`.

Zamówienia JOIN-ują się teraz z WŁAŚCIWĄ HISTORYCZNĄ wersją wymiaru -- po dacie
zamówienia, a NIE zawsze z najnowszą -- dzięki czemu raporty historyczne pozostają
poprawne nawet po przeprowadzce klienta.

> 🏭 **Type 2 to WIĘCEJ wierszy w wymiarze**
>
> Wymiar SCD2 rośnie z każdą zmianą -- klient, który zmienił miasto 3 razy, ma TERAZ 3 wiersze w `dim_klienci` (jeden nieaktualny, potem kolejny nieaktualny, wreszcie jeden aktualny). To nadal WIELOKROTNIE mniejsze niż tabela faktów, więc koszt jest akceptowalny -- ale zapytania MUSZĄ pamiętać o JOIN-owaniu po dacie, nie tylko po kluczu klienta.

<a id="sec4"></a>
## 4. SCD Type 2 przez `dbt snapshot`

Ręczne pisanie logiki Type 2 w czystym SQL-u (`MERGE`, `UPDATE` + `INSERT`) jest
żmudne i podatne na błędy -- dbt ma do tego GOTOWY mechanizm: **`dbt snapshot`**.
Definiujesz snapshot w bloku `{% snapshot ... %}{% endsnapshot %}`, z dwiema
najważniejszymi strategiami wykrywania zmian:

- **`strategy='timestamp'`** -- porównuje kolumnę `updated_at` ze źródła (szybkie,
  wymaga wiarygodnego znacznika czasu w źródle).
- **`strategy='check'`** -- porównuje WSKAZANE kolumny (`check_cols=[...]`) między
  bieżącym stanem a ostatnią wersją w snapshocie -- działa nawet bez `updated_at`,
  kosztem sprawdzania większej liczby kolumn przy każdym uruchomieniu.

In [ ]:
snapshot_kod = '''
{% snapshot snap_klienci %}
{{
    config(
        target_schema='main',
        unique_key='klient_id',
        strategy='check',
        check_cols=['miasto', 'segment'],
    )
}}
select * from {{ ref('stg_klienci') }}
{% endsnapshot %}
'''
with open(f"{DBT_DIR}/snapshots/snap_klienci.sql", "w") as f:
    f.write(snapshot_kod)

# Pierwszy snapshot -- "punkt zerowy" historii (klient #1 ma juz Szczecin, z sekcji 2)
dbt("snapshot")

con = duckdb.connect(DUCKDB_PATH)
print(con.execute(
    "SELECT klient_id, miasto, segment, dbt_valid_from, dbt_valid_to "
    "FROM main.snap_klienci WHERE klient_id = 1"
).fetchdf())
con.close()


In [ ]:
# Symulujemy KOLEJNA zmiane -- klient #1 zmienia segment na Premium
con = duckdb.connect(DUCKDB_PATH)
con.execute("UPDATE raw.klienci SET segment = 'Premium' WHERE klient_id = 1")
con.close()

dbt("run", "--select", "stg_klienci")   # odswiezamy staging PRZED snapshotem
dbt("snapshot")                          # snapshot wykrywa zmiane i DODAJE nowy wiersz

con = duckdb.connect(DUCKDB_PATH)
print(con.execute(
    "SELECT klient_id, miasto, segment, dbt_valid_from, dbt_valid_to "
    "FROM main.snap_klienci WHERE klient_id = 1 ORDER BY dbt_valid_from"
).fetchdf())
con.close()


> ⚡ **`dbt snapshot` MUSI być uruchamiany REGULARNIE**
>
> Snapshot rejestruje TYLKO zmiany, które istniały w źródle W MOMENCIE uruchomienia `dbt snapshot` -- jeśli klient zmienił miasto DWA RAZY między dwoma uruchomieniami, snapshot zobaczy tylko FINALNY stan i przegapi wersję pośrednią. Dlatego w produkcyjnym potoku snapshot uruchamia się regularnie (np. codziennie przez Airflow, Moduł 3), a nie "od czasu do czasu".

In [ ]:
# Poprawny JOIN historyczny: laczymy zamowienie z wersja wymiaru AKTUALNA W MOMENCIE zamowienia
zapytanie = '''
select
    z.zamowienie_id,
    z.data_zamowienia,
    sk.miasto,
    sk.segment
from raw.zamowienia z
inner join main.snap_klienci sk
    on z.klient_id = sk.klient_id
    and z.data_zamowienia >= sk.dbt_valid_from
    and (sk.dbt_valid_to is null or z.data_zamowienia < sk.dbt_valid_to)
where z.klient_id = 1
order by z.data_zamowienia
limit 5
'''
con = duckdb.connect(DUCKDB_PATH)
print(con.execute(zapytanie).fetchdf())
con.close()


<a id="sec5"></a>
## 5. Wzmianka: Type 3 i Type 6

Dla kompletności, dwie rzadziej używane odmiany:

- **Type 3** -- zamiast nowego wiersza, dodajesz KOLUMNĘ `poprzednie_miasto` obok
  `miasto`. Przechowuje TYLKO JEDNĄ poprzednią wartość (nie pełną historię) -- rzadko
  wystarczające, używane głównie gdy interesuje Cię wyłącznie porównanie "przed/po"
  jednej konkretnej zmiany (np. reorganizacji regionów sprzedaży).
- **Type 6** (hybryda 1+2+3) -- łączy Type 2 (pełna historia jako osobne wiersze) z
  dodatkową kolumną "bieżąca wartość" (jak Type 1) widoczną na WSZYSTKICH
  historycznych wierszach danego klienta -- pozwala jednym zapytaniem odpowiedzieć i
  "jakie miasto miał klient WTEDY", i "jakie miasto ma TERAZ", bez dodatkowego joina.

<a id="sec6"></a>
## 6. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- problem zmieniających się atrybutów wymiaru i jego wpływ na raporty historyczne,
- SCD Type 0 (nigdy nie zmieniaj) i Type 1 (nadpisz, domyślne zachowanie zwykłych modeli),
- SCD Type 2 (pełna historia przez nowe wiersze + `dbt_valid_from`/`dbt_valid_to`),
- implementację Type 2 przez `dbt snapshot` (strategie `timestamp` i `check`),
- poprawny JOIN historyczny (po dacie, nie tylko po kluczu),
- krótko: Type 3 (jedna poprzednia wartość) i Type 6 (hybryda).

### 📝 Ćwiczenie 1: Snapshot produktów

Zbuduj snapshot `snap_produkty` (strategia `check`, `check_cols=['kategoria', 'cena']`) dla `stg_produkty` (zbuduj ten staging model, jeśli jeszcze nie istnieje). Uruchom go dwa razy z rzędu BEZ żadnej zmiany danych i sprawdź, że liczba wierszy się NIE zmienia (brak zmiany = brak nowej wersji).

<details>
<summary>Pokaż rozwiązanie</summary>

```python
model_kod = '''
select produkt_id, kategoria, cena from {{ source('raw_dane', 'produkty') }}
'''
with open(f"{DBT_DIR}/models/stg_produkty.sql", "w") as f:
    f.write(model_kod)
dbt("run", "--select", "stg_produkty")

snapshot_kod = '''
{% snapshot snap_produkty %}
{{ config(target_schema='main', unique_key='produkt_id', strategy='check', check_cols=['kategoria', 'cena']) }}
select * from {{ ref('stg_produkty') }}
{% endsnapshot %}
'''
with open(f"{DBT_DIR}/snapshots/snap_produkty.sql", "w") as f:
    f.write(snapshot_kod)

dbt("snapshot", "--select", "snap_produkty")
n1 = duckdb.connect(DUCKDB_PATH).execute("SELECT COUNT(*) FROM main.snap_produkty").fetchone()[0]

dbt("snapshot", "--select", "snap_produkty")
n2 = duckdb.connect(DUCKDB_PATH).execute("SELECT COUNT(*) FROM main.snap_produkty").fetchone()[0]

print(f"Liczba wierszy po 1. uruchomieniu: {n1}, po 2. uruchomieniu: {n2} (bez zmian danych -- powinny byc rowne).")
```

</details>

### 📝 Ćwiczenie 2: Zasymuluj zmianę kategorii produktu

Zmień `kategoria` produktu #1 na `'Ogrod'` (nowa kategoria) w `raw.produkty`, odśwież `stg_produkty`, uruchom `dbt snapshot` ponownie i sprawdź, że `snap_produkty` ma TERAZ dwa wiersze dla `produkt_id = 1` (stary, nieaktualny, i nowy, aktualny).

<details>
<summary>Pokaż rozwiązanie</summary>

```python
con = duckdb.connect(DUCKDB_PATH)
con.execute("UPDATE raw.produkty SET kategoria = 'Ogrod' WHERE produkt_id = 1")
con.close()

dbt("run", "--select", "stg_produkty")
dbt("snapshot", "--select", "snap_produkty")

con = duckdb.connect(DUCKDB_PATH)
print(con.execute(
    "SELECT produkt_id, kategoria, dbt_valid_from, dbt_valid_to FROM main.snap_produkty "
    "WHERE produkt_id = 1 ORDER BY dbt_valid_from"
).fetchdf())
con.close()
```

</details>

### 📝 Ćwiczenie 3: Ile klientów zmieniło segment?

Korzystając z `snap_klienci` (sekcja 4), napisz zapytanie SQL zwracające liczbę UNIKALNYCH `klient_id`, które mają WIĘCEJ NIŻ JEDEN wiersz w `snap_klienci` (czyli zmieniły miasto lub segment przynajmniej raz).

<details>
<summary>Pokaż rozwiązanie</summary>

```python
zapytanie = '''
select count(*) as liczba_klientow_ze_zmiana
from (
    select klient_id
    from main.snap_klienci
    group by klient_id
    having count(*) > 1
)
'''
con = duckdb.connect(DUCKDB_PATH)
print(con.execute(zapytanie).fetchdf())
con.close()
```

</details>

> 🔥 **Wyzwanie: raport przychodu per miasto, POPRAWNY historycznie**
>
> Napisz zapytanie SQL łączące `raw.pozycje_zamowien` + `raw.zamowienia` z `main.snap_klienci` PRAWIDŁOWYM joinem historycznym (jak w sekcji 4), które liczy sumę wartości sprzedaży PER `miasto` -- i porównaj wynik z NAIWNYM podejściem (join tylko po `klient_id`, bez uwzględnienia dat) dla klienta #1, żeby zobaczyć różnicę, jaką robi poprawne wersjonowanie.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
zapytanie_poprawne = '''
select
    sk.miasto,
    round(sum(poz.ilosc * poz.cena_jednostkowa * (1 - poz.rabat)), 2) as przychod
from raw.pozycje_zamowien poz
inner join raw.zamowienia z on poz.zamowienie_id = z.zamowienie_id
inner join main.snap_klienci sk
    on z.klient_id = sk.klient_id
    and z.data_zamowienia >= sk.dbt_valid_from
    and (sk.dbt_valid_to is null or z.data_zamowienia < sk.dbt_valid_to)
where z.klient_id = 1
group by sk.miasto
'''

zapytanie_naiwne = '''
select
    sk.miasto,
    round(sum(poz.ilosc * poz.cena_jednostkowa * (1 - poz.rabat)), 2) as przychod
from raw.pozycje_zamowien poz
inner join raw.zamowienia z on poz.zamowienie_id = z.zamowienie_id
inner join raw.klienci sk on z.klient_id = sk.klient_id
where z.klient_id = 1
group by sk.miasto
'''

con = duckdb.connect(DUCKDB_PATH)
print("Podejscie POPRAWNE (historyczne, rozbite po miastach z roznych okresow):")
print(con.execute(zapytanie_poprawne).fetchdf())
print("Podejscie NAIWNE (caly przychod przypisany do OBECNEGO miasta):")
print(con.execute(zapytanie_naiwne).fetchdf())
con.close()
```

Podejscie naiwne przypisuje CALA historie zakupow klienta #1 do jego OBECNEGO miasta (Szczecin), mimo ze czesc zamowien zlozyl, gdy mieszkal w Gdansku -- to dokladnie ten blad raportowy, ktoremu zapobiega SCD Type 2.

</details>

## Co dalej?

W **Module 8** przechodzimy od świata batch (Airflow, dbt) do świata **streamingu**:
poznajemy koncepcje Apache Kafka -- topiki, partycje, producentów i konsumentów --
fundament przetwarzania danych w czasie rzeczywistym.